# Step 1 - Setup Data

 Creating and Connecting to the Database

In [75]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("superstore.db")

print("Connection created")

Connection created


Loading the Dataset

In [76]:
df = pd.read_csv("Superstore_raw.csv", encoding="latin1")
df.to_sql(
    "Superstore_raw",conn,if_exists="replace",index=False
)
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

df.head()

Rows: 9994
Columns: 21


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


Creating Tables

In [77]:
# Creating customers table

conn.execute("""
CREATE TABLE IF NOT EXISTS customers (

    Customer_ID PRIMARY KEY,
    Customer_Name
)
""")

print("Customers table created successfully!")

# Creating products table

conn.execute("""
CREATE TABLE IF NOT EXISTS products (
    Product_ID PRIMARY KEY,
    Category ,
    Sub_Category ,
    Product_Name 
)
""")

print("Products table created successfully!")

# Creating orders table

conn.execute("""
CREATE TABLE IF NOT EXISTS orders (
    Order_ID ,
    Order_Date ,
    Customer_ID ,
    Product_ID ,
    Sales ,
    Quantity ,
    Profit
)

""")

print("Orders table created successfully!")

Customers table created successfully!
Products table created successfully!
Orders table created successfully!


Checking if tables have been created

In [78]:
tables = pd.read_sql_query("""

SELECT name
FROM sqlite_master
WHERE type='table'

""", conn)

tables

,name
0,customers
1,products
2,orders
3,Superstore_raw


Inserting data from the database

In [79]:
# Inserting data into customers table

conn.execute("""
INSERT OR IGNORE INTO customers SELECT DISTINCT
    [Customer ID],
    [Customer Name]
FROM Superstore_raw
""")
print("Data inserted into customers table successfully!")

# Inserting data into products table

conn.execute("""
INSERT OR IGNORE INTO products SELECT DISTINCT
    [Product ID],
    Category,
    [Sub-Category],
    [Product Name]
FROM Superstore_raw
""")

print("Data inserted into products table successfully!")

# Inserting data into orders table

conn.execute("""
INSERT INTO orders SELECT DISTINCT
    [Order ID],
    [Order Date],
    [Customer ID],
    [Product ID],
    Sales,
    Quantity,
    Profit
FROM Superstore_raw
""")
conn.commit()

print("Data inserted into orders table successfully!")

Data inserted into customers table successfully!
Data inserted into products table successfully!
Data inserted into orders table successfully!


# Step 2 - Performing Queries

Q1.) Find all orders where sales are greater than the average sales. (Subquery)  

In [81]:
# Finding orders whose sales are greater than the average sales

df_q1 = pd.read_sql_query("""
SELECT * FROM orders
WHERE Sales > (SELECT AVG(Sales) FROM orders)
""", conn)

print(f"Orders found: {len(df_q1)}")

df_q1.head()

Orders found: 18872


,Order_ID,Order_Date,Customer_ID,Product_ID,Sales,Quantity,Profit
0,CA-2016-152156,11/8/2016,CG-12520,FUR-BO-10001798,261.9600,2,41.9136
1,CA-2016-152156,11/8/2016,CG-12520,FUR-CH-10000454,731.9400,3,219.5820
2,US-2015-108966,10/11/2015,SO-20335,FUR-TA-10000577,957.5775,5,-383.0310
3,CA-2014-115812,6/9/2014,BH-11710,TEC-PH-10002275,907.1520,6,90.7152
4,CA-2014-115812,6/9/2014,BH-11710,FUR-TA-10001539,1706.1840,9,85.3092


In [82]:
# Checking the average sales value

pd.read_sql_query("""
SELECT AVG(Sales) AS Average_Sales
FROM orders
""", conn)

,Average_Sales
0,229.852846


 **Insight** - 
This query uses a subquery to calculate the average sales value across all orders.
then it returns only those orders whose sales exceed that average, helping identify higher-value transactions within the dataset.

Q.2) Find the highest sales order for each customer. (Subquery)  

In [83]:
#Cretaing index for faster query execution

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_customer_id
ON orders(Customer_ID)
""")
conn.execute("""
CREATE INDEX IF NOT EXISTS idx_order_id
ON orders(Order_ID)
""")
conn.commit()

print("Indexes created successfully!")

Indexes created successfully!


In [84]:
# Finding the highest sales order for each customer

df_q2 = pd.read_sql_query("""
SELECT * FROM orders o
WHERE Sales =(SELECT MAX(Sales) FROM orders
              WHERE Customer_ID = o.Customer_ID)
ORDER BY Customer_ID
""", conn)

print(f"Records found: {len(df_q2)}")
df_q2

Records found: 6360


,Order_ID,Order_Date,Customer_ID,Product_ID,Sales,Quantity,Profit
0,CA-2016-103982,3/3/2016,AA-10315,OFF-SU-10000151,3930.072,3,-786.0144
1,CA-2016-103982,3/3/2016,AA-10315,OFF-SU-10000151,3930.072,3,-786.0144
2,CA-2016-103982,3/3/2016,AA-10315,OFF-SU-10000151,3930.072,3,-786.0144
3,CA-2016-103982,3/3/2016,AA-10315,OFF-SU-10000151,3930.072,3,-786.0144
4,CA-2016-103982,3/3/2016,AA-10315,OFF-SU-10000151,3930.072,3,-786.0144
...,...,...,...,...,...,...,...
6355,CA-2016-152471,7/8/2016,ZD-21925,TEC-PH-10002824,823.960,5,51.4975
6356,CA-2016-152471,7/8/2016,ZD-21925,TEC-PH-10002824,823.960,5,51.4975
6357,CA-2016-152471,7/8/2016,ZD-21925,TEC-PH-10002824,823.960,5,51.4975
6358,CA-2016-152471,7/8/2016,ZD-21925,TEC-PH-10002824,823.960,5,51.4975


In [85]:
# validation for one customer

pd.read_sql_query("""
SELECT Customer_ID, Sales
FROM orders
WHERE Customer_ID = 'AA-10315'
ORDER BY Sales DESC
LIMIT 10
""", conn)

,Customer_ID,Sales
0,AA-10315,3930.072
1,AA-10315,3930.072
2,AA-10315,3930.072
3,AA-10315,3930.072
4,AA-10315,3930.072
5,AA-10315,3930.072
6,AA-10315,3930.072
7,AA-10315,3930.072
8,AA-10315,673.568
9,AA-10315,673.568



**Insight** - The query returned 795 records, which corresponds to the highest sales
order for each customer. Manual verification on sample customers confirmed that the
returned sales value matches the maximum sales value present in the orders table.

Q.3) Calculate total sales for each customer. (CTE)  

In [86]:
# Calculating total sales for each customer using CTE

df_q3 = pd.read_sql_query("""
WITH CustomerSales AS (SELECT Customer_ID, SUM(Sales) AS Total_Sales
FROM orders
GROUP BY Customer_ID)
                          
SELECT * FROM CustomerSales
ORDER BY Total_Sales DESC
""", conn)

print(f"Records found: {len(df_q3)}")

df_q3.head()

Records found: 793


,Customer_ID,Total_Sales
0,SM-20320,200344.400
1,TC-20980,152417.744
2,RB-19360,120938.712
3,TA-21385,116764.960
4,AB-10105,115788.568


In [87]:
# Validation

pd.read_sql_query("""
SELECT Customer_ID, SUM(Sales) AS Total_Sales
FROM orders
GROUP BY Customer_ID
ORDER BY Total_Sales DESC
LIMIT 5
""", conn)

,Customer_ID,Total_Sales
0,SM-20320,200344.400
1,TC-20980,152417.744
2,RB-19360,120938.712
3,TA-21385,116764.960
4,AB-10105,115788.568


**Insight** - This query uses a Common Table Expression (CTE) to calculate the total sales
generated by each customer. The results help identify high-value customers and provide
a clear summary of customer-wise revenue contribution.

Q.4) Find customers whose total sales are above average. (CTE + Subquery)  

In [88]:
# Finding customers whose total sales are above average

df_q4 = pd.read_sql_query("""
WITH CustomerSales AS (SELECT Customer_ID, SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY Customer_ID)
                          
SELECT * FROM CustomerSales WHERE Total_Sales > (SELECT AVG(Total_Sales)
    FROM CustomerSales
)
ORDER BY Total_Sales DESC
""", conn)

print(f"Records found: {len(df_q4)}")

df_q4.head()

Records found: 294


,Customer_ID,Total_Sales
0,SM-20320,200344.400
1,TC-20980,152417.744
2,RB-19360,120938.712
3,TA-21385,116764.960
4,AB-10105,115788.568


In [89]:
# validation 

pd.read_sql_query("""
WITH CustomerSales AS (SELECT Customer_ID, SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY Customer_ID)
                  
SELECT * FROM CustomerSales
ORDER BY Total_Sales DESC
LIMIT 10
""", conn)

,Customer_ID,Total_Sales
0,SM-20320,200344.400
1,TC-20980,152417.744
2,RB-19360,120938.712
3,TA-21385,116764.960
4,AB-10105,115788.568
5,KL-16645,113401.832
6,SC-20095,113138.672
7,HL-15040,102986.384
8,SE-20110,97675.504
9,CC-12370,97032.576



**Insight** - This query combines a Common Table Expression (CTE) and a subquery.
The CTE calculates the total sales for each customer, while the subquery computes
the average of those totals. The final result returns only customers whose total
sales exceed the overall average customer sales.

Q.5) Rank all customers based on total sales. (Window Function)

In [90]:
# Ranking all customers based on total sales

df_q5 = pd.read_sql_query("""
WITH CustomerSales AS (SELECT Customer_ID, SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY Customer_ID)
SELECT Customer_ID, Total_Sales,
    RANK() OVER (ORDER BY Total_Sales DESC) AS Sales_Rank
FROM CustomerSales
ORDER BY Sales_Rank
""", conn)

print(f"Records found: {len(df_q5)}")

df_q5.head()

Records found: 793


,Customer_ID,Total_Sales,Sales_Rank
0,SM-20320,200344.400,1
1,TC-20980,152417.744,2
2,RB-19360,120938.712,3
3,TA-21385,116764.960,4
4,AB-10105,115788.568,5


In [91]:
# Validation

pd.read_sql_query("""
WITH CustomerSales AS (SELECT Customer_ID,SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY Customer_ID)

SELECT Customer_ID,Total_Sales
FROM CustomerSales
ORDER BY Total_Sales DESC
LIMIT 10
""", conn)

#The customers with the highest Total_Sales should have the best ranks (1, 2, 3, ...).

,Customer_ID,Total_Sales
0,SM-20320,200344.400
1,TC-20980,152417.744
2,RB-19360,120938.712
3,TA-21385,116764.960
4,AB-10105,115788.568
5,KL-16645,113401.832
6,SC-20095,113138.672
7,HL-15040,102986.384
8,SE-20110,97675.504
9,CC-12370,97032.576


**Insight** - This query uses a Window Function (RANK) to assign a sales rank to each
customer based on their total sales. Customers with higher total sales receive
better ranks, making it easy to identify top-performing customers.

Q.6) Assign row numbers to each order within a customer. (Window Function + PARTITION BY)  

In [92]:
# Assigning row numbers to each order within a customer

df_q6 = pd.read_sql_query("""

SELECT Customer_ID,Order_ID,Sales,
    ROW_NUMBER() OVER(PARTITION BY Customer_ID ORDER BY Sales DESC) AS Order_Number
FROM orders

""", conn)

print(f"Records found: {len(df_q6)}")

df_q6

Records found: 79944


,Customer_ID,Order_ID,Sales,Order_Number
0,AA-10315,CA-2016-103982,3930.072,1
1,AA-10315,CA-2016-103982,3930.072,2
2,AA-10315,CA-2016-103982,3930.072,3
3,AA-10315,CA-2016-103982,3930.072,4
4,AA-10315,CA-2016-103982,3930.072,5
...,...,...,...,...
79939,ZD-21925,CA-2014-143336,8.560,68
79940,ZD-21925,CA-2014-143336,8.560,69
79941,ZD-21925,CA-2014-143336,8.560,70
79942,ZD-21925,CA-2014-143336,8.560,71


In [93]:
#validation

pd.read_sql_query("""
SELECT Customer_ID,Order_ID,Sales,
    ROW_NUMBER() OVER(PARTITION BY Customer_ID ORDER BY Sales DESC) AS Order_Number
FROM orders
WHERE Customer_ID = 'CG-12520'

""", conn)

,Customer_ID,Order_ID,Sales,Order_Number
0,CG-12520,CA-2016-152156,731.940,1
1,CG-12520,CA-2016-152156,731.940,2
2,CG-12520,CA-2016-152156,731.940,3
3,CG-12520,CA-2016-152156,731.940,4
4,CG-12520,CA-2016-152156,731.940,5
5,CG-12520,CA-2016-152156,731.940,6
6,CG-12520,CA-2016-152156,731.940,7
7,CG-12520,CA-2016-152156,731.940,8
8,CG-12520,CA-2016-152156,261.960,9
9,CG-12520,CA-2016-152156,261.960,10


**Insight**
The ROW_NUMBER() function assigns a unique number to each order within a customer group.

PARTITION BY Customer_ID resets the numbering for every customer, while ORDER BY Sales DESC arranges the orders from highest to lowest sales.

Q.7) Display top 3 customers based on total sales. (Window Function)  

In [94]:
# Displaying top 3 customers based on total sales

df_q7 = pd.read_sql_query("""
WITH customer_sales AS (SELECT Customer_ID,SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY Customer_ID),
                          
ranked_customers AS(SELECT Customer_ID,Total_Sales,RANK() OVER(ORDER BY Total_Sales DESC) AS Sales_Rank
    FROM customer_sales)

SELECT * FROM ranked_customers
WHERE Sales_Rank <= 3
""", conn)

print(f"Top customers found: {len(df_q7)}")

df_q7

Top customers found: 3


,Customer_ID,Total_Sales,Sales_Rank
0,SM-20320,200344.400,1
1,TC-20980,152417.744,2
2,RB-19360,120938.712,3


In [95]:
pd.read_sql_query("""
SELECT Customer_ID,SUM(Sales) AS Total_Sales
FROM orders
GROUP BY Customer_ID
ORDER BY Total_Sales DESC
LIMIT 10
""", conn)

,Customer_ID,Total_Sales
0,SM-20320,200344.400
1,TC-20980,152417.744
2,RB-19360,120938.712
3,TA-21385,116764.960
4,AB-10105,115788.568
5,KL-16645,113401.832
6,SC-20095,113138.672
7,HL-15040,102986.384
8,SE-20110,97675.504
9,CC-12370,97032.576


**Insight**
The query first calculates total sales for each customer and then uses the RANK() window function to rank customers based on their sales contribution.

Customers with the three highest sales ranks are displayed, helping identify the most valuable customers in the dataset.

## Step 3 - Final Combined query

Write one final query that shows: 

Customer Name  

Total Sales  

Rank  

(Use JOIN + CTE + Window Function together) 

In [96]:
# Displaying customer name, total sales and rank

df_final = pd.read_sql_query("""
WITH customer_sales AS(SELECT Customer_ID,SUM(Sales) AS Total_Sales
    FROM orders
    GROUP BY Customer_ID
)

SELECT c.Customer_Name,cs.Total_Sales,
    RANK() OVER(ORDER BY cs.Total_Sales DESC) AS Customer_Rank
FROM customer_sales cs
JOIN customers c
ON cs.Customer_ID = c.Customer_ID
ORDER BY Customer_Rank
""", conn)

print(f"Records found: {len(df_final)}")

df_final.head(10)

Records found: 793


,Customer_Name,Total_Sales,Customer_Rank
0,Sean Miller,200344.400,1
1,Tamara Chand,152417.744,2
2,Raymond Buch,120938.712,3
3,Tom Ashbrook,116764.960,4
4,Adrian Barton,115788.568,5
5,Ken Lonsdale,113401.832,6
6,Sanjit Chand,113138.672,7
7,Hunter Lopez,102986.384,8
8,Sanjit Engle,97675.504,9
9,Christopher Conant,97032.576,10


**Insight**

This query combines a CTE, JOIN, and Window Function to generate a ranked customer sales report.

The CTE calculates total sales for each customer, the JOIN retrieves customer names, and the RANK() function assigns rankings based on total sales. This provides a clear view of the highest-performing customers in the dataset.

In [97]:
# Closing the SQLite connection

conn.close()

print("Database connection closed")

Database connection closed
